In [6]:
import streamlit as st
from pathlib import Path
from langchain.agents import create_sql_agent
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_types import AgentType
from langchain.callbacks import StreamlitCallbackHandler
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from sqlalchemy import create_engine
import sqlite3
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

mysql_user = "admin"
mysql_password = "admin"
mysql_host = "localhost"
mysql_db = "student"

llm=ChatGroq(model_name="Llama3-8b-8192", streaming=True)

database = SQLDatabase(create_engine(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_db}"))

In [8]:
toolkit = SQLDatabaseToolkit(db=database, llm=llm)

agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    agent_type = AgentType.ZERO_SHOT_REACT_DESCRIPTION
)

In [ ]:
agent.invoke("how many records i have in student databse?")

In [11]:
from langchain.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
chain = create_sql_query_chain(llm, database)
response = chain.invoke({"question": "How many sudents are there"})
response

'SELECT COUNT(`NAME`) AS `TOTAL_STUDENTS` FROM `STUDENT`'

In [12]:
database.run(response)

'[(5,)]'

In [13]:
chain.get_prompts()[0].pretty_print()

You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of the S

## Execute SQL query

In [17]:
from langchain_community.tools import QuerySQLDataBaseTool

execute_query = QuerySQLDataBaseTool(db=database)

write_query = create_sql_query_chain(llm, database)

chain = write_query | execute_query

chain.invoke({"question": "How many students enrolled in devops class"})

'[(2,)]'

### Answering the question

In [18]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question. ALWAYS Use all data in database.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

answer = answer_prompt | llm | StrOutputParser()
chain = (
    RunnablePassthrough.assign(query=write_query).assign(
        result=itemgetter("query") | execute_query
    )
    | answer
)

chain.invoke({"question": "How many employees are there"})

'There are 5 employees in the database.'

# Agents

In [44]:
from langchain_community.agent_toolkits import create_sql_agent
from langchain_openai import ChatOpenAI

mysql_user = "root"
mysql_password = "admin"
mysql_host = "localhost"
mysql_db = "school_db"

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# database2 = SQLDatabase(create_engine(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_db}"))
# print(database2)

from langchain_community.utilities import SQLDatabase

database = SQLDatabase.from_uri(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_db}")
print(database)



agent_sql = create_sql_agent(llm, db=database, agent_type="openai-tools", verbose=True)

In [ ]:
agent_sql.invoke(
    "List all the total sales per all student discipline. Which discipline's students spent the most?"
)

In [ ]:
agent_sql.invoke(
    "who are students involved in science and how much did they spend?"
)

In [ ]:
agent_sql.invoke(
    "show me disciplines with corresponding enrolled students in them with amount of money spent"
)

In [ ]:
agent_sql.invoke(
    "who is the most skilled student based classes and grades"
)

# Using a dynamic few-shot prompt

In [52]:
examples = [ 
    {
        "input": "Find the total number of disciplines.",
        "query": "SELECT COUNT(*) FROM DISCIPLINE;",
    },
    {
        "input": "List the total sales per student discipline. Which discipline's students spent the most?",
        "query": """  
                    SELECT 
                        D.NAME AS Discipline,
                        SUM(S.MONEY_SPENT) AS Total_Sales
                    FROM 
                        DISCIPLINE_JOINT DJ
                    INNER JOIN 
                        STUDENT S ON DJ.STUDENT_ID = S.STUDENT_ID
                    INNER JOIN 
                        DISCIPLINE D ON DJ.DISCIPLINE_ID = D.DISCIPLINE_ID
                    GROUP BY 
                        D.NAME
                    ORDER BY 
                        Total_Sales DESC;  

                """
    },
    {
        "input": "How many students are there",
        "query": 'SELECT COUNT(*) FROM "STUDENT"',
    },
]

##### SemanticSimilarityExampleSelector, which will perform a semantic search using the embeddings and vector store we configure to find the examples most similar to our input

In [80]:
from langchain_community.vectorstores import FAISS
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    FAISS,
    k=5,
    input_keys=["input"],
)

In [81]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
    SystemMessagePromptTemplate,
)

system_prefix = """You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the given tools. Only use the information returned by the tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.

If the question does not seem related to the database, just return "I don't know" as the answer.

Here are some examples of user inputs and their corresponding SQL queries:"""

few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=PromptTemplate.from_template(
        "User input: {input}\nSQL query: {query}"
    ),
    input_variables=["input", "dialect", "top_k"],
    prefix=system_prefix,
    suffix="",
)

In [82]:
full_prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate(prompt=few_shot_prompt),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

#### Example formatted prompt

In [ ]:
prompt_val = full_prompt.invoke(
    {
        "input": "How many students are there",
        "top_k": 5,
        "dialect": "mysql",
        "agent_scratchpad": [],
    }
)
print(prompt_val.to_string())

#### Creating our agent with our custom prompt

In [83]:
agent = create_sql_agent(
    llm=llm,
    db=database,
    prompt=full_prompt,
    verbose=False,
    agent_type="openai-tools",
)

##### With misspelled student name it will Fails to find results

In [84]:
agent.invoke({"input": "in how many disciplines kresh ptel is involved?"})

{'input': 'in how many disciplines kresh ptel is involved?',
 'output': 'Kresh Ptel is not involved in any disciplines.'}

## Dealing with high-cardinality columns
Selfcorrecting on misspelled field values (student name, discipline name,.. etc)

High-cardinality of a column is the ability of a field to have many possible values, unlike Enums

In [71]:
import ast
import re


def query_as_list(db, query):
    res = db.run(query)
    res = [el for sub in ast.literal_eval(res) for el in sub if el]
    res = [re.sub(r"\b\d+\b", "", string).strip() for string in res]
    return list(set(res))


students = query_as_list(database, "SELECT NAME FROM STUDENT")
disciplines = query_as_list(database, "SELECT NAME FROM DISCIPLINE")

In [72]:
from langchain.agents.agent_toolkits import create_retriever_tool

vector_db = FAISS.from_texts(students + disciplines, OpenAIEmbeddings())
retriever = vector_db.as_retriever(search_kwargs={"k": 5})
description = """Use to look up values to filter on. Input is an approximate spelling of the proper noun, output is \
valid proper nouns. Use the noun most similar to the search."""
retriever_tool = create_retriever_tool(
    retriever,
    name="search_proper_nouns",
    description=description,
)

In [76]:
system = """You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the given tools. Only use the information returned by the tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.

If you need to filter on a proper noun, you must ALWAYS first look up the filter value using the "search_proper_nouns" tool! 

You have access to the following tables: {table_names}

If the question does not seem related to the database, just return "I don't know" as the answer."""

prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{input}"), MessagesPlaceholder("agent_scratchpad")]
)
agent = create_sql_agent(
    llm=llm,
    db=database,
    extra_tools=[retriever_tool],
    prompt=prompt,
    agent_type="openai-tools",
    verbose=False,
)

### Now it will selfcorrect and Find results even with misspelled student name

In [77]:
agent.invoke({"input": "in how many disciplines kresh ptel is involved?"})

{'input': 'in how many disciplines kresh ptel is involved?',
 'output': 'Krish Patel is involved in 2 disciplines.'}